# Bulletin blueprint — W19 banking demo

Test E2E pipeline: data card → script → audio → render. Sample data inline (`SAMPLE_DATA`, `RAW_FROM_CLAUDE`) — thay bằng data thật + paste-từ-Claude sau.

**Yêu cầu trước khi chạy:** `.env` có `VBEE_TOKEN` + `VBEE_APPID`. Cell 6 cần Node.js + Remotion components (chưa có → Cell 6 sẽ fail intentional).


In [1]:
# Cell 1: Imports + config
import sys
sys.path.insert(0, '..')

import json
import datetime
from pathlib import Path

from lib import schema, validator, vbee, audio, cache, render

PROJECT_ROOT = Path('..').resolve()
TEMPLATE = "bulletin"

# Inputs - đổi mỗi video mới
VIDEO_ID = "2026-05-12_daily_eod"
SESSION_DATE = "2026-05-12"
TOPIC = "Tổng kết phiên giao dịch VBSE Daily EOD"

VOICE = "s_hochiminh_female_vyquangcao_advertise_vc"  # Vy quảng cáo, SG female, advertise (energetic)
SPEED = 1.0

# Setup paths
OUTPUT_DIR = PROJECT_ROOT / "outputs" / VIDEO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = PROJECT_ROOT / "remotion" / "public" / "runs" / VIDEO_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"RUN_DIR      = {RUN_DIR}")


PROJECT_ROOT = D:\twan_projects\video-report
OUTPUT_DIR   = D:\twan_projects\video-report\outputs\2026-05-12_daily_eod
RUN_DIR      = D:\twan_projects\video-report\remotion\public\runs\2026-05-12_daily_eod


d:\twan_projects\video-report\python\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Cell 2 — Build data card

TODO Phase 2: thay `SAMPLE_DATA` bằng `data_card.build(week=WEEK)` đọc từ JSON file thật.

In [2]:
# Cell 2: Build data card from SAMPLE_DATA — daily EOD (VBSE 12/05/2026)
SAMPLE_DATA = {
    "date": SESSION_DATE,
    "vnindex": {
        "open": 1893.36, "high": 1906.64, "low": 1875.25, "close": 1901.10,
        "change_pts": 5.60, "change_pct": 0.30,
        "ma20": 1847.24, "poc": 1865.60, "pivot": 1802.82,
        "volume_shares": 630_830_592, "value_billion": 18759, "liquidity_pct": 77.84,
    },
    "foreign": {
        "buy_value_billion": 2306.41, "sell_value_billion": -3142.11, "net_value_billion": -835.70,
        "buy_volume_million": 56.24, "sell_volume_million": -89.11, "net_volume_million": -32.88,
        "top_buy": [
            {"ticker": "VIC", "value_billion": 195.56},
            {"ticker": "VRE", "value_billion": 119.51},
            {"ticker": "GEX", "value_billion": 96.78},
        ],
        "top_sell": [
            {"ticker": "FPT", "value_billion": 171.95},
            {"ticker": "VHM", "value_billion": 154.35},
            {"ticker": "MSB", "value_billion": 144.36},
        ],
    },
    "sectors": {
        "top_gainers": [
            {"name": "Kinh doanh Bảo hiểm", "change_pct": 2.63, "vsi": 88.68},
            {"name": "Nhựa và Bao bì", "change_pct": 1.20, "vsi": 92.33},
            {"name": "Dịch vụ Dầu khí", "change_pct": 1.13, "vsi": 107.20},
        ],
        "top_losers": [
            {"name": "Hạ tầng Tiện ích", "change_pct": -1.00, "vsi": 59.46},
            {"name": "Dệt may Xuất khẩu", "change_pct": -0.54, "vsi": 104.06},
        ],
    },
}


def build_data_card(d: dict) -> str:
    v = d["vnindex"]
    f = d["foreign"]
    s = d["sectors"]
    lines = [
        f"# Bản tin tổng kết phiên — {d['date']}",
        "",
        "## VN-Index",
        f"- Đóng cửa: **{v['close']:.2f}** điểm ({v['change_pts']:+.2f} / {v['change_pct']:+.2f}%)",
        f"- Open/High/Low: {v['open']:.2f} / {v['high']:.2f} / {v['low']:.2f}",
        f"- MA20: {v['ma20']:.2f}, POC: {v['poc']:.2f}, PIVOT: {v['pivot']:.2f}",
        f"- Thanh khoản: {v['liquidity_pct']:.2f}%, KLGD {v['volume_shares']:,} CP, GTGD {v['value_billion']:,} tỷ",
        "",
        "## Khối ngoại",
        f"- Mua/bán ròng giá trị: {f['net_value_billion']:+.2f} tỷ (mua {f['buy_value_billion']:.2f} / bán {f['sell_value_billion']:.2f})",
        f"- Mua/bán ròng khối lượng: {f['net_volume_million']:+.2f} triệu CP",
        "- Top mua ròng: " + ", ".join(f"{b['ticker']} {b['value_billion']:.1f}t" for b in f["top_buy"]),
        "- Top bán ròng: " + ", ".join(f"{b['ticker']} {b['value_billion']:.1f}t" for b in f["top_sell"]),
        "",
        "## Ngành",
        "- Top tăng: " + ", ".join(f"{g['name']} {g['change_pct']:+.2f}%" for g in s["top_gainers"]),
        "- Top giảm: " + ", ".join(f"{g['name']} {g['change_pct']:+.2f}%" for g in s["top_losers"]),
    ]
    return "\n".join(lines)


card_md = build_data_card(SAMPLE_DATA)
(OUTPUT_DIR / "data_card.md").write_text(card_md, encoding="utf-8")
print(card_md)
print("---")
print(f"Saved: {OUTPUT_DIR / 'data_card.md'}")


# Bản tin tổng kết phiên — 2026-05-12

## VN-Index
- Đóng cửa: **1901.10** điểm (+5.60 / +0.30%)
- Open/High/Low: 1893.36 / 1906.64 / 1875.25
- MA20: 1847.24, POC: 1865.60, PIVOT: 1802.82
- Thanh khoản: 77.84%, KLGD 630,830,592 CP, GTGD 18,759 tỷ

## Khối ngoại
- Mua/bán ròng giá trị: -835.70 tỷ (mua 2306.41 / bán -3142.11)
- Mua/bán ròng khối lượng: -32.88 triệu CP
- Top mua ròng: VIC 195.6t, VRE 119.5t, GEX 96.8t
- Top bán ròng: FPT 171.9t, VHM 154.3t, MSB 144.4t

## Ngành
- Top tăng: Kinh doanh Bảo hiểm +2.63%, Nhựa và Bao bì +1.20%, Dịch vụ Dầu khí +1.13%
- Top giảm: Hạ tầng Tiện ích -1.00%, Dệt may Xuất khẩu -0.54%
---
Saved: D:\twan_projects\video-report\outputs\2026-05-12_daily_eod\data_card.md


## Cell 3 — PASTE script JSON từ Claude

Flow thật: copy `data_card.md` ở Cell 2 → Claude.ai Project `Video — Bulletin` → Claude trả JSON → paste vào đây.

Now: dùng `RAW_FROM_CLAUDE` đã viết tay (đúng format Claude sẽ trả) để demo.

In [3]:
# Cell 3: PASTE script JSON từ Claude (raw string giữa """ """)
# Bulletin daily EOD 5 scenes — VBSE 12/05/2026.
# Scene types: intro, metric, ranking, outro.
# - Mọi info trong narration phải có visual tương ứng (highlights/items).
# - appear_at_sec sync với narration để bullet xuất hiện đúng lúc giọng đọc tới.
# - "!" cho nhấn, "?" cho rising tone, em-dash "—" cho ngắt nhịp, "nhé!" cuối câu.
RAW_FROM_CLAUDE = """
{
  "video_id": "2026-05-12_daily_eod",
  "template": "bulletin",
  "meta": {
    "title": "VBSE Daily EOD - 12/05/2026",
    "voice": "s_hochiminh_female_vyquangcao_advertise_vc",
    "speed": 1.0,
    "fps": 30,
    "width": 1080,
    "height": 1080,
    "created_at": "2026-05-12T16:00:00+07:00"
  },
  "scenes": [
    {
      "id": "s01",
      "type": "intro",
      "narration_text": "Chào mừng các bạn đến với bản tin chứng khoán VBSE! Phiên giao dịch ngày mười hai tháng năm — kết thúc với sắc xanh. VN Index chốt phiên tại một nghìn chín trăm linh một điểm! Tăng năm phẩy sáu điểm, tương đương không phẩy ba phần trăm. Thanh khoản đạt mười tám nghìn bảy trăm năm mươi chín tỷ đồng. Với hơn sáu trăm ba mươi triệu cổ phiếu sang tay!",
      "data": {
        "headline": "VN Index 1.901,10",
        "category": "Tổng kết phiên",
        "issue_label": "Phiên giao dịch 12/05/2026",
        "highlights": [
          {"text": "VN Index 1.901,10 điểm", "appear_at_sec": 4.5, "style": "bullet"},
          {"text": "+0,30%", "appear_at_sec": 9.0, "style": "stat"},
          {"text": "Thanh khoản 18.759 tỷ đồng", "appear_at_sec": 13.5, "style": "bullet"},
          {"text": "KLGD 630 triệu CP", "appear_at_sec": 17.5, "style": "bullet"}
        ]
      }
    },
    {
      "id": "s02",
      "type": "metric",
      "narration_text": "Tín hiệu kỹ thuật đáng chú ý! VN Index hiện đang giao dịch — trên đường trung bình hai mươi phiên. Cao hơn MA hai mươi tới năm mươi ba phẩy chín điểm. Khẳng định xu hướng tăng vẫn đang được duy trì vững vàng!",
      "data": {
        "metric": "VN Index vs MA20",
        "current_value": 1901.10,
        "previous_value": 1847.24,
        "unit": " điểm",
        "delta": 53.86,
        "context_note": "MA20: 1.847,24 — xu hướng tăng được duy trì"
      }
    },
    {
      "id": "s03",
      "type": "ranking",
      "narration_text": "Top ba nhóm ngành tăng mạnh nhất phiên hôm nay! Dẫn đầu — Kinh doanh Bảo hiểm, tăng hai phẩy sáu mươi ba phần trăm. Vị trí thứ hai — Nhựa và Bao bì, một phẩy hai mươi phần trăm. Và thứ ba — Dịch vụ Dầu khí, một phẩy mười ba phần trăm!",
      "data": {
        "title": "Top ngành tăng giá",
        "summary_text": "Phiên 12/05 — sắc xanh lan toả",
        "items": [
          {"rank": 1, "label": "Kinh doanh Bảo hiểm", "change_pct": 2.63, "appear_at_sec": 2.5},
          {"rank": 2, "label": "Nhựa và Bao bì", "change_pct": 1.20, "appear_at_sec": 7.5},
          {"rank": 3, "label": "Dịch vụ Dầu khí", "change_pct": 1.13, "appear_at_sec": 11.5}
        ]
      }
    },
    {
      "id": "s04",
      "type": "ranking",
      "narration_text": "Chiều ngược lại — áp lực bán ròng từ khối ngoại! Tổng giá trị bán ròng đạt tám trăm ba mươi sáu tỷ đồng. Đứng đầu danh sách là FPT — bán ròng một trăm bảy mươi hai tỷ. Tiếp theo Vinhomes, bị bán một trăm năm mươi tư tỷ. Và MSB — một trăm bốn mươi tư tỷ đồng.",
      "data": {
        "title": "Khối ngoại bán ròng",
        "summary_text": "Tổng giá trị: -835,7 tỷ đồng",
        "items": [
          {"rank": 1, "label": "FPT", "value": -171.95, "value_suffix": "tỷ", "appear_at_sec": 5.0},
          {"rank": 2, "label": "VHM", "value": -154.35, "value_suffix": "tỷ", "appear_at_sec": 10.0},
          {"rank": 3, "label": "MSB", "value": -144.36, "value_suffix": "tỷ", "appear_at_sec": 14.5}
        ]
      }
    },
    {
      "id": "s05",
      "type": "outro",
      "narration_text": "Đó là tóm tắt phiên giao dịch ngày mười hai tháng năm! Hẹn gặp lại các bạn — với phân tích kỹ thuật cuối tuần. Đừng quên theo dõi VBSE để cập nhật sớm nhất nhé!",
      "data": {
        "cta": "Theo dõi VBSE",
        "handle": "@vbse",
        "next_topic": "Phân tích kỹ thuật cuối tuần",
        "date_label": "Phiên 12/05/2026"
      }
    }
  ]
}
"""

print(f"RAW length: {len(RAW_FROM_CLAUDE)} chars")


RAW length: 3786 chars


## Cell 4 — Parse + validate

Layer 1 (Pydantic) + Layer 3 (content sanity). Nếu fail, error message paste lại Claude để sửa.

In [4]:
# Cell 4: Parse + validate, loop nếu fail
try:
    raw_dict = validator.parse_claude_output(RAW_FROM_CLAUDE)
    script = schema.Script.model_validate(raw_dict)
    errors = validator.check_content_sanity(script)

    if errors:
        msg = validator.format_errors_for_claude(errors)
        print(msg)
        raise ValueError("Validate fail - paste msg trên cho Claude fix")

    (OUTPUT_DIR / "script_final.json").write_text(
        script.model_dump_json(indent=2), encoding="utf-8"
    )
    print(f"OK: {len(script.scenes)} scenes")
    for s in script.scenes:
        print(f"  [{s.id}] {s.type}: {s.narration_text[:60]}...")
except Exception as e:
    print(f"LỖI: {e}")
    raise


OK: 5 scenes
  [s01] intro: Chào mừng các bạn đến với bản tin chứng khoán VBSE! Phiên gi...
  [s02] metric: Tín hiệu kỹ thuật đáng chú ý! VN Index hiện đang giao dịch —...
  [s03] ranking: Top ba nhóm ngành tăng mạnh nhất phiên hôm nay! Dẫn đầu — Ki...
  [s04] ranking: Chiều ngược lại — áp lực bán ròng từ khối ngoại! Tổng giá tr...
  [s05] outro: Đó là tóm tắt phiên giao dịch ngày mười hai tháng năm! Hẹn g...


## Cell 5 — Enrich audio (Vbee + cache + postprocess)

Lần đầu chạy: ~5-15s/scene qua Vbee API. Lần sau: cache hit qua hash narration text.

In [5]:
# Cell 5: Enrich audio
client = vbee.VbeeClient()
audio_dir = RUN_DIR / "audio"
audio_dir.mkdir(parents=True, exist_ok=True)

for scene in script.scenes:
    audio_p = cache.audio_path(scene, audio_dir, voice=VOICE, speed=SPEED)
    if not audio_p.exists():
        print(f"[{scene.id}] Synthesizing ({len(scene.narration_text)} chars)...")
        client.synthesize(scene.narration_text, audio_p, voice=VOICE, speed=SPEED)
        audio.trim_silence(audio_p)
        audio.normalize_peak(audio_p)
    else:
        print(f"[{scene.id}] Cache hit: {audio_p.name}")
    scene.audio_path = f"audio/{audio_p.name}"
    scene.duration = audio.get_duration(audio_p)

# Save enriched script for Remotion to read
(RUN_DIR / "script.json").write_text(
    script.model_dump_json(indent=2), encoding="utf-8"
)

total = sum(s.duration for s in script.scenes)
print(f"\nTotal: {total:.1f}s, {len(script.scenes)} scenes ready")
print(f"Saved: {RUN_DIR / 'script.json'}")


[s01] Synthesizing (349 chars)...
[s02] Synthesizing (208 chars)...
[s03] Synthesizing (234 chars)...
[s04] Synthesizing (258 chars)...
[s05] Synthesizing (160 chars)...

Total: 58.5s, 5 scenes ready
Saved: D:\twan_projects\video-report\remotion\public\runs\2026-05-12_daily_eod\script.json


## Cell 6 — Render Remotion

**BLOCKED** đến khi: (1) cài Node.js, (2) `cd remotion && npm install`, (3) viết Remotion bulletin Composition + 4 scene components.

In [6]:
# Cell 6: Render Remotion (sẽ fail nếu Node/Remotion chưa setup)
output_path = render.render_video(
    template=TEMPLATE,
    video_id=VIDEO_ID,
    project_root=PROJECT_ROOT,
)
print(f"Render xong: {output_path}")

Render xong: D:\twan_projects\video-report\outputs\2026-05-12_daily_eod\render.mp4


## Cell 7 — Preview inline

In [7]:
# Cell 7: Preview
from IPython.display import Video
Video(str(output_path), width=360)


## Cell 8 — Save manifest

In [8]:
# Cell 8: Manifest
manifest = {
    "video_id": VIDEO_ID,
    "template": TEMPLATE,
    "topic": TOPIC,
    "session_date": SESSION_DATE,
    "voice": VOICE,
    "speed": SPEED,
    "scenes_count": len(script.scenes),
    "total_duration_sec": round(sum(s.duration for s in script.scenes), 3),
    "created_at": datetime.datetime.now().astimezone().isoformat(),
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "video_id": "2026-05-12_daily_eod",
  "template": "bulletin",
  "topic": "Tổng kết phiên giao dịch VBSE Daily EOD",
  "session_date": "2026-05-12",
  "voice": "s_hochiminh_female_vyquangcao_advertise_vc",
  "speed": 1.0,
  "scenes_count": 5,
  "total_duration_sec": 58.512,
  "created_at": "2026-05-12T15:33:47.417863+07:00"
}
